# Module 16: Multi-Agent Commons -- FEP Meets Social Simulation
## Capstone

**Spinning Up in Active Inference** | Notebook 16 of 16

---

This capstone module brings together everything from the curriculum -- the Free Energy Principle, Active Inference, JAX acceleration, hierarchical models -- and applies it to a genuinely important problem: **commons governance**.

The tragedy of the commons (Hardin, 1968) describes how shared resources are depleted when individuals act in self-interest. But Elinor Ostrom (Nobel Prize, 2009) showed that communities *can* sustainably manage commons through institutional design. Her 8 design principles describe the conditions under which cooperation emerges.

We model this using **SustainHub** -- an open-source software community as a digital commons. Each contributor is an Active Inference agent with beliefs about project health, task urgency, and the value of different actions. The experiment ladder (L0-L7) progressively adds AIF concepts to show how each one contributes to cooperative behavior.

By the end of this module you will:
1. Understand the SustainHub POMDP structure
2. Run single-agent and multi-agent AIF simulations
3. Walk the experiment ladder from pure RL (L0) to full FEP integration (L7)
4. See how FEP beliefs format as natural language for LLM consumption
5. Decompose EFE into pragmatic (productivity) and epistemic (community learning)
6. Observe adaptive model re-learning when the environment surprises agents

**Key references:**
- Ostrom, E. (1990). *Governing the Commons.* Cambridge University Press.
- Smith, Friston & Whyte (2022). *A Step-by-Step Tutorial on Active Inference.*
- Rohira, V. (2025). *SustainHub: Multi-Agent Social Simulation.* Concordia.

**Prerequisites:** Modules 1-15 (the entire curriculum).
**Time:** ~90 minutes.

## 16.1 Ostrom's Design Principles for Commons Governance

Before we write any code, let us ground ourselves in the theory. Ostrom identified 8 design principles that characterize long-enduring commons institutions:

| # | Principle | SustainHub Analog |
|---|-----------|-------------------|
| 1 | Clearly defined boundaries | Project scope, contributor roles |
| 2 | Rules match local conditions | Role-specific task preferences |
| 3 | Collective-choice arrangements | Sprint planning (all agents choose) |
| 4 | Monitoring | Harmony Index observation signal |
| 5 | Graduated sanctions | Declining project health -> worse outcomes |
| 6 | Conflict resolution | Belief updating -> consensus |
| 7 | Recognized right to organize | Agents have autonomous decision-making |
| 8 | Nested enterprises | Hierarchical generative models |

The key insight for AIF: **sustainable commons management requires agents that balance exploitation (productivity) with exploration (learning about community state)**. This is exactly what the EFE decomposition provides: pragmatic value = productivity, epistemic value = community learning.

In [ ]:
# -- Setup ------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import sys
sys.path.insert(0, '/Users/mhough/dev/concordia')

# SustainHub Active Inference module
from examples.games.sustain_hub import active_inference as aif
from examples.games.sustain_hub import fep_integration
from examples.games.sustain_hub import experiments

# Curriculum plotting utilities
sys.path.insert(0, '..')
from utils.plotting import COLORS

np.random.seed(42)
print("SustainHub Active Inference loaded.")
print(f"  Health states:  {aif.PROJECT_HEALTH_STATES}")
print(f"  Urgency states: {aif.TASK_URGENCY_STATES}")
print(f"  HI observations: {aif.HI_OBSERVATIONS}")
print(f"  Task observations: {aif.TASK_OBSERVATIONS}")
print(f"  Actions: {aif.ACTIONS}")

## 16.2 The SustainHub POMDP

SustainHub models an open-source software project as a **Partially Observable Markov Decision Process (POMDP)**:

- **Hidden states** (not directly observable):
  - Project health: `healthy`, `stressed`, `declining` (3 states)
  - Task urgency: `balanced`, `bugs_critical`, `docs_neglected`, `review_backlog` (4 states)

- **Observations** (what agents perceive):
  - Harmony Index level: `high`, `medium`, `low` (noisy signal of health)
  - Task outcome: `success`, `partial`, `failure` (noisy signal of action effectiveness)

- **Actions** (what agents can do):
  - `bug_fix`, `feature`, `documentation`, `code_review`, `mentor`, `skip`

The generative model (A, B, C, D) encodes how the project works:
- A: how health/urgency produce observations (likelihood)
- B: how actions change health/urgency (transitions)
- C: what outcomes agents prefer (preferences)
- D: prior beliefs about initial state

In [ ]:
# -- Examine the generative model matrices ----------------------------------

A_matrices = aif.build_A_matrix()
B_matrices = aif.build_B_matrix()
C_matrices = aif.build_C_matrix(role_preference='contributor')
D_matrices = aif.build_D_matrix(health_prior='uncertain')

print("A-matrix (HI observation | health, urgency):")
print(f"  Shape: {A_matrices[0].shape}")
print(f"  When healthy+balanced:   P(high|.)={A_matrices[0][0,0,0]:.2f}, "
      f"P(medium|.)={A_matrices[0][1,0,0]:.2f}, P(low|.)={A_matrices[0][2,0,0]:.2f}")
print(f"  When declining+balanced: P(high|.)={A_matrices[0][0,2,0]:.2f}, "
      f"P(medium|.)={A_matrices[0][1,2,0]:.2f}, P(low|.)={A_matrices[0][2,2,0]:.2f}")

print(f"\nB-matrix (health transitions per action):")
print(f"  Shape: {B_matrices[0].shape}")
print(f"  bug_fix:  healthy->healthy = {B_matrices[0][0,0,0]:.2f}, "
      f"declining->healthy = {B_matrices[0][0,2,0]:.2f}")
print(f"  feature:  healthy->healthy = {B_matrices[0][0,0,1]:.2f}, "
      f"declining->healthy = {B_matrices[0][0,2,1]:.2f}")
print(f"  skip:     healthy->healthy = {B_matrices[0][0,0,5]:.2f}, "
      f"declining->healthy = {B_matrices[0][0,2,5]:.2f}")

print(f"\nC-matrix (preferences):")
print(f"  HI pref:   {C_matrices[0]}  (prefer high HI)")
print(f"  Task pref: {C_matrices[1]}  (prefer success)")

print(f"\nD-matrix (prior beliefs):")
print(f"  Health: {D_matrices[0].round(3)}")
print(f"  Urgency: {D_matrices[1].round(3)}")

In [ ]:
# -- Visualize the A-matrix as a heatmap ------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A1: HI observation given health (averaged over urgency)
A1_avg = A_matrices[0].mean(axis=2)  # average over urgency
im0 = axes[0].imshow(A1_avg, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
axes[0].set_title('A-matrix: P(HI observation | health)\n(averaged over urgency)')
axes[0].set_xlabel('Health State')
axes[0].set_ylabel('HI Observation')
axes[0].set_xticks(range(3))
axes[0].set_xticklabels(aif.PROJECT_HEALTH_STATES)
axes[0].set_yticks(range(3))
axes[0].set_yticklabels(aif.HI_OBSERVATIONS)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# B1: Health transitions for each action (from healthy state)
actions_to_show = [0, 1, 3, 5]  # bug_fix, feature, code_review, skip
transition_data = np.zeros((len(actions_to_show), 3))  # action x next_health
for i, a_idx in enumerate(actions_to_show):
    transition_data[i, :] = B_matrices[0][:, 0, a_idx]  # from healthy

im1 = axes[1].imshow(transition_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('B-matrix: P(next health | healthy, action)')
axes[1].set_xlabel('Next Health State')
axes[1].set_ylabel('Action')
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(aif.PROJECT_HEALTH_STATES)
axes[1].set_yticks(range(len(actions_to_show)))
axes[1].set_yticklabels([aif.ACTIONS[a] for a in actions_to_show])
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.suptitle('SustainHub Generative Model', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 16.3 Single Agent: The Observe-Decide-Learn Loop

Let us create a single `ActiveInferenceAgent` and walk through one sprint cycle:
1. **Observe**: receive a noisy HI signal and task outcome
2. **Decide**: compute EFE for all actions, select via softmax
3. **Learn**: update habits (E-vector) based on outcome valence

In [ ]:
# -- Create a single agent and run the observe-decide-learn loop ------------

agent = aif.ActiveInferenceAgent(
    name='Priya',
    role='contributor',
    gamma=1.0,
    alpha=16.0,
    learning_rate=0.1,
    health_prior='uncertain',
)

print(f"Agent: {agent.name} (role: {agent.role})")
print(f"  Gamma (precision): {agent.gamma}")
print(f"  Alpha (action precision): {agent.alpha}")
print(f"  Prior beliefs (health): {agent.beliefs[0].round(3)}")
print(f"  Prior beliefs (urgency): {agent.beliefs[1].round(3)}")
print(f"  Habits (E): {dict(zip(aif.ACTIONS, agent.E.round(3)))}")

# Step 1: Observe
print("\n--- Step 1: OBSERVE ---")
agent.observe('medium', 'success')
print(f"  Observed: HI=medium, task=success")
print(f"  Updated beliefs (health): {agent.beliefs[0].round(3)}")
print(f"  Updated beliefs (urgency): {agent.beliefs[1].round(3)}")

# Step 2: Decide
print("\n--- Step 2: DECIDE ---")
action, probs = agent.decide()
print(f"  Action probabilities:")
for i, a in enumerate(aif.ACTIONS):
    marker = " <--" if a == action else ""
    print(f"    {a:<15} {probs[i]:.4f}{marker}")
print(f"  Selected action: {action}")

# Step 3: Learn
print("\n--- Step 3: LEARN ---")
agent.learn(valence=0.5)  # positive outcome
print(f"  Valence: +0.5 (positive outcome)")
print(f"  Updated habits: {dict(zip(aif.ACTIONS, agent.E.round(3)))}")

## 16.4 Multi-Agent Simulation: 6 Agents, Different Roles

In a real commons, agents have different roles, different expertise, and therefore different beliefs about what the community needs. Let us create 6 agents with diverse roles and run them through multiple sprints.

In [ ]:
# -- Create 6 agents with different roles -----------------------------------

agent_configs = [
    ('Raj',    'maintainer',       'pessimistic'),
    ('Lin',    'contributor',      'uncertain'),
    ('Yuki',   'innovator',        'optimistic'),
    ('Elena',  'knowledge_curator','uncertain'),
    ('Marcus', 'contributor',      'uncertain'),
    ('Anya',   'maintainer',       'uncertain'),
]

agents = []
for name, role, prior in agent_configs:
    a = aif.ActiveInferenceAgent(
        name=name, role=role, gamma=1.0, alpha=16.0,
        learning_rate=0.1, health_prior=prior,
    )
    agents.append(a)

# Run 5 sprints
n_sprints = 5
sprint_results = []

# Simulate environment
true_health = 0  # healthy
true_urgency = 0  # balanced
rng = np.random.RandomState(42)

for sprint in range(n_sprints):
    # Stress event at sprint 2
    if sprint == 2:
        true_health = 1  # stressed
        true_urgency = 1  # bugs_critical

    # Generate observation
    A1 = aif.build_A_matrix()[0]
    obs_probs = A1[:, true_health, true_urgency]
    obs_probs = obs_probs / obs_probs.sum()
    hi_idx = rng.choice(3, p=obs_probs)
    hi_obs = aif.HI_OBSERVATIONS[hi_idx]

    sprint_data = {'sprint': sprint, 'hi_obs': hi_obs, 'actions': {},
                   'beliefs': {}, 'true_health': aif.PROJECT_HEALTH_STATES[true_health]}

    for agent in agents:
        agent.observe(hi_obs, 'success')
        action, probs = agent.decide()
        reward = 2.0 if action in ['bug_fix', 'code_review'] and true_urgency == 1 else 1.0
        agent.learn((reward - 1.0) / 2.0)

        sprint_data['actions'][agent.name] = action
        sprint_data['beliefs'][agent.name] = {
            'health': agent.beliefs[0].round(3).tolist(),
            'urgency': agent.beliefs[1].round(3).tolist(),
        }

    sprint_results.append(sprint_data)

    # Environment dynamics
    bug_fixes = sum(1 for a in sprint_data['actions'].values() if a == 'bug_fix')
    if bug_fixes >= 2:
        true_health = max(0, true_health - 1)

    print(f"Sprint {sprint} | HI={hi_obs:<6} | Health={sprint_data['true_health']:<10} | "
          f"Actions: {dict(sprint_data['actions'])}")

In [ ]:
# -- Visualize beliefs across agents: different roles -> different beliefs ---

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, agent in enumerate(agents):
    ax = axes[idx // 3, idx % 3]

    # Extract health beliefs over time
    health_beliefs = []
    for sprint_data in sprint_results:
        hb = sprint_data['beliefs'][agent.name]['health']
        health_beliefs.append(hb)

    health_beliefs = np.array(health_beliefs)
    for s_idx, s_name in enumerate(aif.PROJECT_HEALTH_STATES):
        ax.plot(range(n_sprints), health_beliefs[:, s_idx], 'o-',
                linewidth=2, markersize=6, label=s_name)

    ax.set_title(f"{agent.name} ({agent.role})")
    ax.set_xlabel('Sprint')
    ax.set_ylabel('Belief')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.axvline(2, color='red', linestyle='--', alpha=0.3)  # stress event

plt.suptitle('Health Beliefs Across Agents\n(Red line = stress event at sprint 2)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Agents with 'pessimistic' priors (Raj) converge to stressed beliefs faster")
print("  - Agents with 'optimistic' priors (Yuki) are slower to update")
print("  - All agents eventually converge toward the true state (Bayesian consistency)")

## 16.5 The Experiment Ladder: L0 through L4

The experiment ladder progressively adds Active Inference concepts:

| Level | Name | New Concept | RL Analog |
|-------|------|-------------|----------|
| L0 | Pure RL Baseline | Reward signal | Q-learning |
| L1 | + Prediction Error | Surprise tracking | TD error |
| L2 | + Epistemic Value | Information gain | UCB exploration |
| L3 | + Belief Updating | Variational inference | State estimation |
| L4 | + EFE Policy | Expected Free Energy | Q-value policy |

Let us run levels 0-4 and compare.

In [ ]:
# -- Run experiment ladder levels 0-4 ---------------------------------------

ladder_results = []

for level_idx in range(5):  # L0 through L4
    level = experiments.EXPERIMENT_LEVELS[level_idx]
    runner = experiments.ExperimentRunner(
        level=level,
        num_agents=6,
        num_sprints=5,
        seed=42,
    )
    result = runner.run()
    ladder_results.append(result)

    print(f"L{level_idx}: {result['level_name']:<30} | "
          f"Mean HI={result['mean_hi']:.4f} | "
          f"Coverage={result['mean_coverage']:.4f} | "
          f"RQ={result['resilience_quotient']:.4f}")

In [ ]:
# -- Plot: Harmony Index progression across levels --------------------------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# HI trajectories
colors_ladder = plt.cm.viridis(np.linspace(0, 1, 5))
for i, result in enumerate(ladder_results):
    ax1.plot(result['hi_trajectory'], 'o-', color=colors_ladder[i],
             linewidth=2, markersize=8, label=f"L{i}: {result['level_name'][:20]}")

ax1.set_xlabel('Sprint')
ax1.set_ylabel('Harmony Index')
ax1.set_title('HI Trajectory Across Experiment Levels')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.axvline(2, color='red', linestyle='--', alpha=0.3, label='Stress event')

# Summary bar chart
level_names = [f"L{i}" for i in range(5)]
mean_his = [r['mean_hi'] for r in ladder_results]
coverages = [r['mean_coverage'] for r in ladder_results]

x = np.arange(5)
ax2.bar(x - 0.15, mean_his, 0.3, label='Mean HI', color=COLORS.get('aif', '#4CAF50'))
ax2.bar(x + 0.15, coverages, 0.3, label='Coverage', color=COLORS.get('rl', '#2196F3'))
ax2.set_xticks(x)
ax2.set_xticklabels(level_names)
ax2.set_ylabel('Score')
ax2.set_title('Summary Metrics by Level')
ax2.legend()
ax2.grid(alpha=0.3, axis='y')

plt.suptitle('Experiment Ladder: How AIF Concepts Improve Commons Governance',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 16.6 FEPAgentComponent: Bridging AIF and LLM

The `FEPAgentComponent` from `fep_integration.py` formats Active Inference beliefs and EFE decompositions as natural language context that can be injected into an LLM prompt. This is the key to the L7 "LLM-as-Node" level: the FEP computation informs the LLM, and the LLM's decision is interpreted as a policy selection.

Let us see what this context looks like.

In [ ]:
# -- FEPAgentComponent: format beliefs for LLM consumption ------------------

fep_agent = fep_integration.FEPAgentComponent(
    name='Priya',
    role='contributor',
    gamma=1.0,
    alpha=16.0,
    adaptive=True,
)

# Process an observation
obs_result = fep_agent.process_observation(
    text="Sprint report: 3 bugs fixed, 1 feature shipped. HI at 0.65.",
    hi_level='medium',
    task_outcome='success',
)

print("Observation processing result:")
print(f"  Composite obs index: {obs_result.get('obs_idx', 'N/A')}")
print(f"  Surprise: {obs_result.get('surprise', 'N/A'):.3f} nats")
print(f"  Surprising? {obs_result.get('is_surprising', False)}")

# Compute EFE for all actions
efe_values = fep_agent.compute_efe_all_actions()
print(f"\nEFE per action:")
for i, a in enumerate(aif.ACTIONS):
    print(f"  {a:<15} G = {efe_values[i]:.4f}")

# Get the recommended action
rec_action_idx, rec_probs = fep_agent.get_recommended_action()
print(f"\nRecommended action: {aif.ACTIONS[rec_action_idx]}")
print(f"Action probabilities: {dict(zip(aif.ACTIONS, rec_probs.round(4)))}")

In [ ]:
# -- Show the FEP context string that gets injected into LLM prompts --------

fep_context = fep_agent.get_fep_context()

print("=" * 70)
print("FEP CONTEXT (injected into LLM prompt):")
print("=" * 70)
print(fep_context)
print("=" * 70)
print("\nThis natural language summary of beliefs, surprise, and action")
print("recommendations gets prepended to the LLM's decision prompt.")
print("The LLM can then reason about the FEP analysis in natural language.")

## 16.7 EFE Decomposition for Commons: Pragmatic vs. Epistemic

The EFE decomposition reveals *why* the agent prefers certain actions. In the commons context:

- **Pragmatic value** = expected productivity (does this action improve the Harmony Index?)
- **Epistemic value** = expected information gain (does this action help me learn about community health?)

An agent that only has pragmatic value will exploit its preferred task type. An agent with epistemic value will sometimes explore unfamiliar tasks to learn about the project state.

In [ ]:
# -- EFE decomposition for commons ------------------------------------------

decomp = fep_agent.compute_efe_decomposition()

print("EFE Decomposition for Contributor Agent:")
print(f"{'Action':<15} {'G_total':>10} {'Pragmatic':>12} {'Epistemic':>12} "
      f"{'% Epistemic':>12}")
print("-" * 65)

for a_name in aif.ACTIONS:
    g_total = decomp[a_name]['G_total']
    prag = decomp[a_name]['pragmatic']
    epist = decomp[a_name]['epistemic']
    total = abs(prag) + abs(epist)
    pct = abs(epist) / total * 100 if total > 0 else 0
    print(f"{a_name:<15} {g_total:>10.4f} {prag:>12.4f} {epist:>12.4f} {pct:>11.1f}%")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

pragmatic_vals = [decomp[a]['pragmatic'] for a in aif.ACTIONS]
epistemic_vals = [decomp[a]['epistemic'] for a in aif.ACTIONS]

x = np.arange(aif.NUM_ACTIONS)
ax.bar(x - 0.15, pragmatic_vals, 0.3, label='Pragmatic (productivity)',
       color=COLORS.get('rl', '#2196F3'))
ax.bar(x + 0.15, epistemic_vals, 0.3, label='Epistemic (community learning)',
       color=COLORS.get('aif', '#4CAF50'))
ax.set_xlabel('Action')
ax.set_ylabel('Value')
ax.set_title('EFE Decomposition: Why the Agent Prefers Certain Actions')
ax.set_xticks(x)
ax.set_xticklabels(aif.ACTIONS, rotation=45, ha='right')
ax.legend()
ax.axhline(0, color='k', linewidth=0.5)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 16.8 Adaptive Generative Model: Re-Learning Under Surprise

The `AdaptiveGenerativeModel` in `fep_integration.py` can re-learn its A and B matrices when the `FEPBeliefTracker` detects persistently high surprise. This models an agent that realizes "my model of the world is wrong" and updates accordingly.

This is the FEP at work: minimize free energy not just by updating beliefs (inference) but by updating the model itself (learning).

In [ ]:
# -- Adaptive model: show surprise tracking and model updates ----------------

# Create a FEPBeliefTracker
tracker = fep_integration.FEPBeliefTracker(surprise_threshold=3.0)

# Create a simple agent for demonstration
demo_agent = fep_integration.FEPAgentComponent(
    name='Demo',
    role='contributor',
    gamma=1.0,
    alpha=16.0,
    adaptive=True,
)

# Simulate a sequence of observations: first normal, then surprising
observation_sequence = [
    ('high', 'success'),    # Expected -- project is healthy
    ('high', 'success'),    # Expected
    ('medium', 'success'),  # Slightly surprising
    ('low', 'failure'),     # Very surprising! Something changed.
    ('low', 'failure'),     # Still surprising
    ('low', 'partial'),     # Consistent with new state
    ('medium', 'partial'),  # Recovery?
]

surprises = []
health_beliefs = []

print(f"{'Step':<5} {'HI':<8} {'Task':<8} {'Surprise':>10} {'Surprising?':>12} "
      f"{'P(healthy)':>11} {'P(stressed)':>12} {'P(declining)':>13}")
print("-" * 85)

for step, (hi, task) in enumerate(observation_sequence):
    result = demo_agent.process_observation(
        text=f"Sprint {step}: HI={hi}, outcome={task}",
        hi_level=hi,
        task_outcome=task,
    )

    surprise = result.get('surprise', 0.0)
    is_surprising = result.get('is_surprising', False)
    surprises.append(surprise)

    # Get current health beliefs
    beliefs = result.get('health_beliefs', np.array([0.33, 0.33, 0.34]))
    health_beliefs.append(beliefs)

    print(f"{step:<5} {hi:<8} {task:<8} {surprise:>10.3f} {'YES' if is_surprising else 'no':>12} "
          f"{beliefs[0]:>11.3f} {beliefs[1]:>12.3f} {beliefs[2]:>13.3f}")

# Plot surprise trajectory
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(surprises, 'ro-', linewidth=2, markersize=8)
ax1.axhline(3.0, color='red', linestyle='--', alpha=0.5, label='Surprise threshold')
ax1.set_ylabel('Surprise (nats)')
ax1.set_title('Bayesian Surprise Over Time')
ax1.legend()
ax1.grid(alpha=0.3)

health_beliefs_arr = np.array(health_beliefs)
for s_idx, s_name in enumerate(aif.PROJECT_HEALTH_STATES):
    ax2.plot(health_beliefs_arr[:, s_idx], 'o-', linewidth=2, markersize=6, label=s_name)
ax2.set_xlabel('Step')
ax2.set_ylabel('Belief')
ax2.set_title('Health Beliefs Adapt to Surprising Observations')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Adaptive Model: Surprise Triggers Belief Revision',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 16.9 The Full Experiment Ladder: Conceptual Bridge

Let us examine the full experiment ladder definition to see how each level maps RL concepts to AIF concepts. This is the curriculum's ultimate Rosetta Stone -- applied to a real social simulation.

In [ ]:
# -- Print the full experiment ladder mapping --------------------------------

print("THE EXPERIMENT LADDER: RL -> ACTIVE INFERENCE")
print("=" * 90)

for level in experiments.EXPERIMENT_LEVELS:
    print(f"\nLevel {level.level}: {level.name}")
    print(f"  New concept:   {level.new_concept}")
    print(f"  RL analog:     {level.rl_analog}")
    print(f"  AIF mechanism: {level.aif_mechanism}")

    features = []
    if level.use_reward_signals: features.append('reward')
    if level.use_prediction_error: features.append('pred_error')
    if level.use_epistemic_value: features.append('epistemic')
    if level.use_belief_updating: features.append('beliefs')
    if level.use_efe_policy: features.append('EFE')
    if level.use_habit_learning: features.append('habits')
    if level.use_precision_dynamics: features.append('precision')
    if level.use_llm_nodes: features.append('LLM-node')
    print(f"  Active features: [{', '.join(features)}]")

print("\n" + "=" * 90)
print("Each level builds cumulatively on the previous.")
print("L0 is pure RL. L7 is full AIF with LLM integration.")
print("The progression shows how each AIF concept adds value.")

In [ ]:
# -- Run levels 0-6 and create comparison table -----------------------------

all_results = []

for level_idx in range(7):  # L0 through L6 (L7 needs LLM)
    level = experiments.EXPERIMENT_LEVELS[level_idx]
    runner = experiments.ExperimentRunner(
        level=level,
        num_agents=6,
        num_sprints=5,
        seed=42,
    )
    result = runner.run()
    all_results.append(result)

# Print comparison table
experiments.print_comparison_table(all_results)

In [ ]:
# -- Comprehensive visualization of the ladder results ----------------------

fig = plt.figure(figsize=(18, 12))
gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# Panel 1: HI trajectories for all levels
ax1 = fig.add_subplot(gs[0, 0])
colors_all = plt.cm.viridis(np.linspace(0, 1, len(all_results)))
for i, result in enumerate(all_results):
    ax1.plot(result['hi_trajectory'], 'o-', color=colors_all[i],
             linewidth=2, markersize=5, label=f"L{i}", alpha=0.8)
ax1.set_xlabel('Sprint')
ax1.set_ylabel('Harmony Index')
ax1.set_title('HI Trajectories Across All Levels')
ax1.legend(fontsize=8, ncol=2)
ax1.grid(alpha=0.3)

# Panel 2: Mean HI bar chart
ax2 = fig.add_subplot(gs[0, 1])
level_labels = [f"L{i}" for i in range(len(all_results))]
mean_his_all = [r['mean_hi'] for r in all_results]
bars = ax2.bar(level_labels, mean_his_all, color=colors_all)
ax2.set_ylabel('Mean Harmony Index')
ax2.set_title('Mean HI by Experiment Level')
ax2.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, mean_his_all):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

# Panel 3: Coverage and diversity
ax3 = fig.add_subplot(gs[1, 0])
coverages_all = [r['mean_coverage'] for r in all_results]
diversities_all = [r['mean_diversity'] for r in all_results]
x_all = np.arange(len(all_results))
ax3.bar(x_all - 0.15, coverages_all, 0.3, label='Coverage', color='#2196F3')
ax3.bar(x_all + 0.15, diversities_all, 0.3, label='Diversity', color='#4CAF50')
ax3.set_xticks(x_all)
ax3.set_xticklabels(level_labels)
ax3.set_ylabel('Score')
ax3.set_title('Task Coverage and Action Diversity')
ax3.legend()
ax3.grid(alpha=0.3, axis='y')

# Panel 4: Resilience quotient
ax4 = fig.add_subplot(gs[1, 1])
rqs = [r['resilience_quotient'] for r in all_results]
bars_rq = ax4.bar(level_labels, rqs, color=colors_all)
ax4.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='RQ=1 (no degradation)')
ax4.set_ylabel('Resilience Quotient')
ax4.set_title('Resilience: HI After Stress / HI Before Stress')
ax4.legend()
ax4.grid(alpha=0.3, axis='y')

plt.suptitle('Experiment Ladder: Comprehensive Results\n'
             'Each level adds one AIF concept. More concepts = better commons governance.',
             fontsize=14, y=1.02)
plt.show()

## 16.10 Capstone Exercise: Design a Novel Commons Scenario

You now have all the tools to design your own commons governance simulation. Choose a scenario and adapt the SustainHub framework:

### Option A: **Fishery Management**
- **Hidden states:** fish population (abundant, moderate, depleted) x season (spawning, migration, dormant)
- **Observations:** catch size (large, medium, small) x water quality (clear, murky, algae)
- **Actions:** fish_heavily, fish_moderately, rest, patrol, restock
- **Key tension:** fishing depletes the population; resting allows recovery
- **Ostrom mapping:** principle 2 (rules match local conditions -- fishing limits by season)

### Option B: **Community Water Rights**
- **Hidden states:** aquifer level (high, medium, low) x rainfall (wet, normal, dry)
- **Observations:** well output (strong, normal, weak) x neighbor reports (cooperative, neutral, competitive)
- **Actions:** irrigate_full, irrigate_half, conserve, share, report
- **Key tension:** individual irrigation depletes the shared aquifer
- **Ostrom mapping:** principle 4 (monitoring -- agents observe neighbor behavior)

### Option C: **Knowledge Commons (Wikipedia-style)**
- **Hidden states:** article quality (featured, good, stub) x editor morale (high, mixed, burned_out)
- **Observations:** page views (rising, stable, falling) x edit wars (none, minor, major)
- **Actions:** write, edit, review, discuss, revert, rest
- **Key tension:** individual contributions improve quality; edit wars degrade morale
- **Ostrom mapping:** principle 6 (conflict resolution -- discuss vs. revert)

For your chosen scenario:
1. Define the A, B, C, D matrices
2. Create agents with role-specific preferences
3. Run the observe-decide-learn loop for 10 sprints
4. Plot the EFE decomposition: when does epistemic value dominate?
5. Compare L0 (pure RL) vs. L4 (EFE policy) on your scenario

In [ ]:
# -- Capstone exercise skeleton: design your own commons ---------------------

# Step 1: Define your POMDP structure
# Uncomment and fill in for your chosen scenario

# SCENARIO = "fishery"  # or "water" or "knowledge"
# 
# HIDDEN_STATES_1 = ['abundant', 'moderate', 'depleted']
# HIDDEN_STATES_2 = ['spawning', 'migration', 'dormant']
# OBSERVATIONS_1 = ['large_catch', 'medium_catch', 'small_catch']
# OBSERVATIONS_2 = ['clear', 'murky', 'algae']
# ACTIONS = ['fish_heavily', 'fish_moderately', 'rest', 'patrol', 'restock']
# 
# # Step 2: Build matrices
# # A1: P(catch | fish_population, season)
# A1 = np.zeros((len(OBSERVATIONS_1), len(HIDDEN_STATES_1), len(HIDDEN_STATES_2)))
# # TODO: fill in the likelihood matrix
# 
# # B1: P(next_population | current_population, action)
# B1 = np.zeros((len(HIDDEN_STATES_1), len(HIDDEN_STATES_1), len(ACTIONS)))
# # TODO: fill in the transition matrix
# 
# # Step 3: Create agents
# # TODO: create agents with different roles (e.g., commercial_fisher, conservationist)
# 
# # Step 4: Run simulation
# # TODO: implement the observe-decide-learn loop

print("Capstone Exercise: Design a Novel Commons Scenario")
print("=" * 50)
print("Choose from: Fishery, Water Rights, or Knowledge Commons")
print("See the instructions above for POMDP structure.")
print("\nHint: Start by adapting the SustainHub matrices.")
print("The key is choosing A/B that create a genuine commons dilemma:")
print("individual benefit vs. collective sustainability.")

## 16.11 Reflecting on the Curriculum

We have come a long way:

| Module | Topic | Key Concept |
|--------|-------|------------|
| 1 | The Behaving Animal | Stimulus-response vs. cognitive maps |
| 2 | Bellman and Value | The Bellman equation: V(s) = max_a [r + gamma*V(s')] |
| 3 | Prediction Errors | TD error: the brain's reward signal |
| 4 | Model-Based RL | Planning with a world model |
| 5 | The Free Energy Principle | Organisms minimize surprise |
| 6 | Generative Models | The POMDP: A, B, C, D matrices |
| 7 | Belief Propagation | Inference as message passing |
| 8 | Active Inference | Action as inference: minimize EFE |
| 9 | The T-Maze | Epistemic foraging: information-seeking behavior |
| 10 | Sequential EFE | Multi-step planning: the value of planning ahead |
| 11 | Model Learning | Learning A/B matrices from data |
| 12 | Deep AIF | Neural network generative models |
| **13** | **The Rosetta Stone** | **RL and AIF are the same math** |
| **14** | **JAX Scaling** | **1000 agents in parallel with vmap** |
| **15** | **Hierarchical AIF** | **Multi-level models, temporal abstraction** |
| **16** | **Multi-Agent Commons** | **FEP meets social simulation (this notebook)** |

The central insight: **Active Inference provides a unified framework for perception, action, and learning.** By decomposing the Expected Free Energy into pragmatic (reward-seeking) and epistemic (information-seeking) components, it reveals why agents explore, how they balance exploitation with learning, and how multiple agents can sustainably manage shared resources.

The SustainHub experiment ladder shows this progression concretely: each AIF concept (prediction error, epistemic value, belief updating, EFE, habits, precision, LLM integration) adds measurable value to commons governance. The FEP is not just a theory -- it is a practical framework for building better agents.

---

## Further Reading

1. **Ostrom, E. (1990).** *Governing the Commons: The Evolution of Institutions for Collective Action.* Cambridge University Press. The foundational work on commons governance.

2. **Smith, R., Friston, K.J., & Whyte, C.J. (2022).** *A Step-by-Step Tutorial on Active Inference and its Application to Empirical Data.* Journal of Mathematical Psychology. The tutorial that inspired our PGMax implementation.

3. **Millidge, B., Tschantz, A., & Buckley, C.L. (2020).** *Whence the Expected Free Energy?* Neural Computation. The paper that proves the RL-AIF equivalence.

4. **Sajid, N., Ball, P.J., Parr, T., & Friston, K.J. (2021).** *Active Inference: Demystified and Compared.* Neural Computation. Clear comparison of AIF with RL, control theory, and Bayesian brain theories.

5. **Eghbal, N. (2020).** *Working in Public: The Making and Maintenance of Open Source Software.* Stripe Press. The inspiration for SustainHub's social dynamics.

6. **Friston, K.J., Rosch, R., Parr, T., Price, C., & Bowman, H. (2017).** *Deep Temporal Models and Active Inference.* Neuroscience & Biobehavioral Reviews. The hierarchical AIF framework.